# modular-diffusion: the steering error of a band engine of Stable Diffusion 1.5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/modular-diffusion/blob/v0.18.0/colab/steering_read.ipynb)

A band engine is trained to give Stable Diffusion 1.5's answer (the predicted noise) on one stretch of the noise range, and its own error is read on held-out rows while it trains. The sampler does not use one answer: with guidance `g` it uses `a(empty) + g (a(c) - a(empty))`, the answer to the empty prompt pushed along the PROMPT DIFFERENCE `d = a(c) - a(empty)`. Since `guided_E - guided_T = (E(empty) - T(empty)) + g (d_E - d_T)`, an engine's error on the prompt difference is multiplied by the guidance in the sampler, while the own-error read weighs it once, and the prompt difference is a small part of an answer. This notebook reads it, with no training and no pictures, on the probe of the trainings' own reads (the same 64 held-out rows, the same noise, the same four steps a stretch), every row answered with its prompt and with the empty prompt, by the part (`E`) and by the model (`T`):

- `R = sum |E(c) - T(c)|^2 / sum |T(c)|^2`, the own error, relative
- `S = sum |d_E - d_T|^2 / sum |d_T|^2`, the steering error
- `G = sum |guided_E - guided_T|^2 / sum |guided_T|^2` at guidance 7.5, the guided error
- `P = sum |d_T|^2 / sum |T(c)|^2`, how large the prompt difference is in the model's answer

each over the stretch and per probe step, beside the error on the empty side and the per-element own error (the training's own read, to the bit).

The parts, one job each: in the middle stretch the stored assembly's middle engine (`2222/nomid/3`, trained) and two untrained reduced copies of the model (`born:2222/nomid/4`, `born:2222/nomid/3`); in the noisy stretch the stored assembly's noisy engine (`tiny`), the whole-range `tiny` engine, the noisy `2222/nomid/3` engine and the seed twin of the noisy `tiny` engine; in the clean stretch the prompt-reading clean engine and the first capture's clean engine without the prompt, the one it replaced in the assembly. In every stretch two controls: the model itself read as a part (it must read exactly 0) and a part of the shape `2222/nomid/3` with random weights (its steering error must be large); in the middle stretch that random part is read twice (the same numbers to the last digit). Every read records its part's weights by their SHA-256.

The cells: (1) install at a release tag; (2) the card, the model and the probe rows, the job list, a smoke of every job on tiny stand-ins (the stored engines' trainings run small first) and one read timed on the real model for the projection; (3) the stored engines brought back from the Hub (the cell stops if one is missing), the job list through one runner, each finished job shipped, the report, and the machine given back only when everything finished was shipped.

RESUME: run the cells again. Finished jobs keep their markers under `OUT`; on a new machine they come back from the Hub, so nothing runs twice.

In [ ]:
# C1 - install at a tag (extras on the name); torch is not a dependency, so the runtime's torch stays
%pip install -q "modular-diffusion[capture,hub,data,guard] @ git+https://github.com/AbstractEyes/modular-diffusion@v0.18.0"
import os, sys, time, json, gc, hashlib, shutil, statistics, threading
import numpy as np
import torch, geolip.modular_diffusion as MD
assert tuple(int(x) for x in MD.__version__.split(".")) >= (0, 18, 0), "RESTART REQUIRED: Runtime > Restart session, then run C1 again"
import diffusers, transformers, huggingface_hub
print("torch", torch.__version__, "| diffusers", diffusers.__version__, "| transformers", transformers.__version__,
      "| huggingface_hub", huggingface_hub.__version__, "| package", MD.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("card:", props.name, f"| memory {props.total_memory / 2**30:.1f} GiB | cpus: {os.cpu_count()}")
else:
    print("card: NONE (choose a GPU runtime)")
OUT = "/content/steering_read"       # the output root; keep it the same to resume
os.makedirs(OUT, exist_ok=True)

The card is printed (inference only: the model and the parts in float32, as the trainings' own reads were made; an L4 is enough). SD1.5 comes at its pinned revision; the probe rows are the first 256 pictures of COCO val2017 with their content hash checked, encoded once by the model's image encoder, and every read takes the first 64 (the trainings' probe). The stored engines are named with the arguments they were trained with; they never run here. The smoke trains small stand-ins of them on the tiny random model, runs every read of the list on them and checks that the model reads exactly 0 as a part, that the random part read twice repeats, that the engine without the prompt has a steering error of exactly 1 and that each stored part's record carries its weight files' SHA-256; it is shipped (a check of the token). Then one read runs on the real model as the roster runs a job, on a quarter of the probe rows, and its seconds give the projection.

In [ ]:
# C2 - the card, the model, the probe rows, the job list, a smoke on tiny stand-ins, one read timed for the projection
from huggingface_hub import snapshot_download, hf_hub_download
import subprocess
from geolip.modular_diffusion import hub, run
from geolip.modular_diffusion.capture import BANDS
from geolip.modular_diffusion.capture.data import center_crop, encode_shards, parquet_pictures, read_index, synthetic_shards
from geolip.modular_diffusion.capture.teacher import load_teacher
from geolip.modular_diffusion.experiments import capture
TEACHER, PROBE, TRAIN, RUNS = f"{OUT}/sd15", f"{OUT}/probe", f"{OUT}/train", f"{OUT}/runs"
REPO = "AbstractPhil/geolip-multirotary-diffusion-collective"   # the results repository ("" = nothing shipped)
PREFIX = "modular_diffusion/capture/three_band"
FIRST_PREFIX = "modular_diffusion/capture/first_capture"
NOT_SHIPPED = ("checkpoint.pt", "SHIPPED", "FROM_HUB")           # the resume checkpoint and two markers; the rest ships
DEVICE = "cuda"
os.makedirs(RUNS, exist_ok=True)

# the card: inference only, the model and every part in float32 (as the trainings' own reads were made)
assert torch.cuda.is_available(), "no card: choose a GPU runtime (an L4 is enough)"
CARD_NAME, CARD_GIB = torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"card: {CARD_NAME} | memory {CARD_GIB:.1f} GiB")

# the model at its pinned revision (its image encoder encodes the probe rows); the probe pictures: the first 256 of
# COCO val2017's first parquet file at a pinned revision, with their first captions and their content hash checked
# (the probe of the band measurement and of every training's own read), encoded once; every read takes the first 64
snapshot_download(repo_id="stable-diffusion-v1-5/stable-diffusion-v1-5", revision="451f4fe16113bff5a5d2269ed5ad43b0592e9a14",
                  allow_patterns=["model_index.json", "scheduler/scheduler_config.json", "tokenizer/*", "text_encoder/config.json",
                                  "text_encoder/model.safetensors", "unet/config.json", "unet/diffusion_pytorch_model.safetensors",
                                  "vae/config.json", "vae/diffusion_pytorch_model.safetensors"], local_dir=TEACHER)
PARQUETS = [hf_hub_download(repo_id="lmms-lab-encoder/COCO-Caption2017", repo_type="dataset", revision="3bdd5827e243cc3084ac69a1111e69c3ab9193ff",
                            filename=f"data/val-0000{i}-of-00002.parquet", local_dir=f"{OUT}/coco") for i in (0,)]       # the first file holds the probe rows
N_PROBE = 256

def content_hash(pictures, captions, size=512):
    h = hashlib.sha256()
    for p, c in zip(pictures, captions):
        a = ((center_crop(p, size) + 1.0) * 127.5).round().clamp(0, 255).to(torch.uint8)    # the array the encoder sees
        h.update(a.numpy().tobytes())
        h.update(c.encode("utf-8") + b"\0")
    return h.hexdigest()

EXPECTED_CONTENT_HASH = "93d333a65f7bbdac6ced9143ff55852273c4e2a64deb7f8f641aeb1878f35c44"           # the 256 probe pictures of the band-needs run and the first capture
ready = lambda folder, n: os.path.exists(f"{folder}/index.json") and sum(read_index(folder)["counts"]) == n
if not ready(PROBE, N_PROBE):
    with_vae = load_teacher(TEACHER, device="cuda", dtype=torch.float32, vae=True)
    pictures_, captions, meta = parquet_pictures(PARQUETS[0], N_PROBE)
    HASH = content_hash(pictures_, captions)
    print("probe content hash", HASH, "| expected", EXPECTED_CONTENT_HASH)
    assert HASH == EXPECTED_CONTENT_HASH, "the probe pictures are not the expected data"
    encode_shards(pictures_, captions, PROBE, with_vae.vae, with_vae.tokenize, shard_size=64, batch=16, size=512, device="cuda")
    json.dump({"content_hash": HASH, "pictures": [{"row": i, "id": m.get("id"), "license": m.get("license")} for i, m in enumerate(meta)]},
              open(f"{PROBE}/pictures.json", "w", encoding="utf-8"), indent=1)
    del pictures_
    del with_vae
    gc.collect()
    torch.cuda.empty_cache()
print("probe shards:", read_index(PROBE)["counts"])

# the stored engines this read is made of, named with the arguments they were trained with (they come back from the
# Hub in the third cell under its assert and never run here; the smoke trains small stand-ins of them): the engines of
# the stored assembly (the prompt-reading clean engine of 2222/nomid/3, the middle engine of 2222/nomid/3, the noisy
# engine of tiny), the whole-range engine of tiny, the noisy engine of 2222/nomid/3, the noisy tiny engine's seed twin,
# and the first capture's clean engine without the prompt (the engine the prompt-reading one replaced in the assembly)
CLEAN_TEXT, MIDDLE, NOISY = "clean_2222nomid3_inherit_text_s0", "middle_2222nomid3_inherit_s0", "noisy_tiny_inherit_s0"
WHOLE, NOISY_2222, NOISY_S1 = "whole_tiny_inherit_9000_s0", "noisy_2222nomid3_inherit_s0", "noisy_tiny_inherit_s1"
FIRST_CLEAN = "clean_2222nomid3_inherit_s0"
HUB_AT = {FIRST_CLEAN: FIRST_PREFIX}      # a stored job that does not live under PREFIX on the Hub
TRAINING = f"variant=band_engine init=inherit text=1 batch=16 lr=5e-5 autocast=bf16 read_every=250 keep_checkpoint=0 data={TRAIN} dtype=fp32"


def stored_jobs():
    return [(CLEAN_TEXT, "variant=band_engine band=clean shape=2222/nomid/3 init=inherit text=1 steps=3000 lr=5e-05 batch=16 "
             f"drop=0.1 seed=0 autocast=bf16 read_every=250 keep_checkpoint=0 dtype=fp32 data={TRAIN} micro=8", [], "train"),
            (MIDDLE, f"{TRAINING} shape=2222/nomid/3 band=middle steps=3000 seed=0", [], "train"),
            (NOISY, f"{TRAINING} shape=tiny band=noisy steps=3000 seed=0", [], "train"),
            (WHOLE, f"{TRAINING} shape=tiny band=middle rho=1.0 steps=9000 seed=0", [], "train"),
            (NOISY_2222, f"{TRAINING} shape=2222/nomid/3 band=noisy steps=3000 seed=0", [], "train"),
            (NOISY_S1, f"{TRAINING} shape=tiny band=noisy steps=3000 seed=1", [], "train"),
            (FIRST_CLEAN, "variant=band_engine band=clean shape=2222/nomid/3 init=inherit text=0 steps=3000 lr=5e-05 batch=16 "
             f"drop=0.1 seed=0 autocast=bf16 read_every=250 keep_checkpoint=0 dtype=fp32 data={TRAIN}", [], "train")]


# the job list: (tag, arguments, the jobs it waits for, kind), one part read on one stretch a job, the stretches in
# this order, each stretch's controls first: the model itself read as a part (it must read exactly 0), a part of the
# shape 2222/nomid/3 with random weights from seed 0 (its steering error must be large), and in the middle stretch the
# same random part read again in a job of its own (it must repeat to the last digit); then the stretch's parts: a
# stored engine's folder (its averaged weights) or born:<shape> (the model's reduced copy, untrained). Every read is
# made on the probe of the trainings' own reads (COMMON: the first 64 probe rows, the noise of seed 1000, four steps a
# stretch, 16 rows a forward), each row answered with its prompt and with the empty prompt, the guided answers at 7.5
COMMON = (f"teacher={TEACHER} device=cuda threads=4 probe_data={PROBE} probe_first=0 probe_rows=64 probe_seed=1000 "
          "probe_per_band=4 chunk=16")
STEERING = "variant=steering guidance=7.5 seed=0"
RANDOM_PART = "random:2222/nomid/3"
STRETCH_PARTS = {"middle": [(MIDDLE, True), ("born:2222/nomid/4", False), ("born:2222/nomid/3", False)],
                 "noisy": [(NOISY, True), (WHOLE, True), (NOISY_2222, True), (NOISY_S1, True)],
                 "clean": [(CLEAN_TEXT, True), (FIRST_CLEAN, True)]}
name_of = lambda part: part.replace("born:", "born_").replace("random:", "random_").replace("/", "")


def job_list(runs):
    jobs = []

    def read(stretch, part, stored=False, tag=None):
        where = f"{runs}/{part}" if stored else part
        jobs.append((tag or f"steer_{name_of(part)}_{stretch}", f"{STEERING} band={stretch} part={where}",
                     [part] if stored else [], "steer"))

    for stretch, parts in STRETCH_PARTS.items():
        read(stretch, "teacher")
        read(stretch, RANDOM_PART)
        if stretch == "middle":
            read(stretch, RANDOM_PART, tag=f"steer_{name_of(RANDOM_PART)}_middle_repeat")
        for part, stored in parts:
            read(stretch, part, stored)
    return jobs


# the smoke: the stored engines' trainings small on the tiny random model (their stand-ins), then every read of the list
# small on them, and the checks that apply: the model read as a part reads exactly 0 in every stretch; the random part
# read twice gives the same numbers to the last digit; the engine without the prompt has a steering error of exactly 1
# (its two answers are one); each stored part's record carries the SHA-256 of its stand-in's two weight files
def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 24), b""):
            h.update(chunk)
    return h.hexdigest()


finite = lambda x: x is not None and bool(np.isfinite(x))
SMOKE = f"{OUT}/smoke_steering"
shutil.rmtree(SMOKE, ignore_errors=True)
synthetic_shards(f"{SMOKE}/rows", rows=8, shard_size=3, size=16, seed=4)       # the tiny model's 16 x 16 latents
SMALL = (f" tiny=1 device={DEVICE} probe_data={SMOKE}/rows probe_rows=4 probe_per_band=2 chunk=2 steps=4 batch=2 read_every=2"
         " save_every=0 log_every=0" + ("" if DEVICE == "cuda" else " autocast=none"))
smoke_jobs = stored_jobs() + job_list(f"{SMOKE}/runs")
S = {}
for t, args, needs, kind in smoke_jobs:
    S[t] = run.main("capture", f"{args} {COMMON}{SMALL}".split() + [f"out={SMOKE}/runs/{t}"])
SR = {t: S[t]["steering"] for t, *_, k in smoke_jobs if k == "steer"}
assert all(SR[f"steer_teacher_{b}"][k] == 0.0 for b in BANDS for k in ("R", "S", "G", "U", "mse")), \
    "the model read as a part does not read exactly 0"
REPEAT = f"steer_{name_of(RANDOM_PART)}_middle"
assert SR[REPEAT] == SR[f"{REPEAT}_repeat"] and S[REPEAT]["weights"] == S[f"{REPEAT}_repeat"]["weights"], \
    "the random part read twice gave other numbers"
assert SR[f"steer_{FIRST_CLEAN}_clean"]["S"] == 1.0, "the engine without the prompt does not read a steering error of 1"
for t, args, needs, kind in smoke_jobs:
    if kind == "steer" and needs:
        rec, folder = S[t]["weights"], f"{SMOKE}/runs/{needs[0]}"
        assert rec["job"] == needs[0] and rec["engine_sha256"] == sha256_of(f"{folder}/engine.safetensors") \
            and rec["engine_avg_sha256"] == sha256_of(f"{folder}/engine_avg.safetensors"), f"{t}: the part's record is not its weights"
assert all(finite(r[k]) for r in SR.values() for k in ("R", "S", "G", "P")), "a read is not finite"
print(f"smoke: {len(smoke_jobs)} jobs on the tiny model ({len(SR)} reads); the model reads exactly 0 as a part in every "
      "stretch, the random part repeats, the engine without the prompt reads a steering error of 1, each stored part's "
      "record carries its weight files' hashes; the random part's steering error on the tiny model: "
      + ", ".join(f"{b} {SR[f'steer_{name_of(RANDOM_PART)}_{b}']['S']:.3g}" for b in BANDS))

if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")          # placed in this process's environment only
    # with the tiny engines' weights, so the upload of large files is checked too
    assert hub.ship(SMOKE, REPO, f"{PREFIX}/_smoke_steering", "the steering error of a part: smoke run", ignore=NOT_SHIPPED), \
        "the smoke folder was not shipped: check the token and the repository before the long run"

# one read of the list's form on the real model, run as the roster runs a job (a new process), on a quarter of the
# probe rows: born:2222/nomid/3 in the middle stretch. Timed: its start-up (the new process, the imports, the model
# loaded), the part's making, and the answers (the model's and the part's, the rows' prompts and the empty prompt).
# A job's cost = the start-up + the making + four times the answers (a stored engine's weights are read from the disk
# besides: two files of about 2 GB for a 2222/nomid/3 engine, not timed here)
JOB_ENV = dict(os.environ, OMP_NUM_THREADS="4", MKL_NUM_THREADS="4", HF_TOKEN="", PYTHONUTF8="1", PYTHONUNBUFFERED="1")


def as_a_job(args, out):
    shutil.rmtree(out, ignore_errors=True)
    os.makedirs(out)
    t = time.time()
    with open(f"{out}/run.log", "wb") as fh:
        p = subprocess.run([sys.executable, "-u", "-m", "geolip.modular_diffusion.run", "capture"] + args.split() + [f"out={out}"],
                           env=JOB_ENV, stdout=fh, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL)
    wall = time.time() - t
    tail = open(f"{out}/run.log", encoding="utf-8", errors="replace").read()[-2000:]
    assert p.returncode == 0, f"the smoke job in {out} failed:\n{tail}"
    return json.load(open(f"{out}/result.json", encoding="utf-8")), wall


TIMED_ROWS, REAL_EXTRA = 16, ""
TIMED, TIMED_WALL = as_a_job(f"{STEERING} band=middle part=born:2222/nomid/3 {COMMON} probe_rows={TIMED_ROWS}{REAL_EXTRA}",
                             f"{OUT}/timed_read")
ANSWERS_S = TIMED["seconds_teacher"] + TIMED["seconds_part"]
MAKING_S = TIMED["seconds_part_load"]
STARTUP_S = max(0.0, TIMED_WALL - ANSWERS_S - MAKING_S)
r16 = TIMED["steering"]
assert all(finite(r16[k]) for k in ("R", "S", "G", "P")) and r16["rows"] == TIMED_ROWS, "the timed read is not a read"
print(f"one read on this card ({TIMED_ROWS} probe rows, the middle stretch, born:2222/nomid/3): R {r16['R']:.4g}, "
      f"S {r16['S']:.4g}, G {r16['G']:.4g}, P {r16['P']:.4g}; the start-up {STARTUP_S:.0f} s, the part made in "
      f"{MAKING_S:.0f} s, the answers {ANSWERS_S:.0f} s")


def cost(args, kind):
    """A job's projected seconds on this card (a stored engine never runs here: 0)."""
    if kind != "steer":
        return 0.0
    return STARTUP_S + MAKING_S + (64 / TIMED_ROWS) * ANSWERS_S


JOBS = stored_jobs() + job_list(RUNS)
COST = {t: cost(a, k) for t, a, _, k in JOBS}
assert len({t for t, *_ in JOBS}) == len(JOBS) and all(n in COST for _, _, ns, _ in JOBS for n in ns)
N_READS = sum(k == "steer" for *_, k in JOBS)
INSTRUMENTS = {"card": CARD_NAME, "card_gib": CARD_GIB, "job_startup_seconds": STARTUP_S, "part_making_seconds": MAKING_S,
               "answers_seconds": ANSWERS_S, "timed_rows": TIMED_ROWS,
               "timed_read": {k: r16[k] for k in ("R", "S", "G", "P", "U", "mse")},
               "smoke_random_part_S": {b: SR[f"steer_{name_of(RANDOM_PART)}_{b}"]["S"] for b in BANDS}}
print(f"the job list: {len(JOBS)} jobs ({N_READS} reads of one part each; the {len(JOBS) - N_READS} stored engines come back "
      f"from the Hub in the third cell and never run), about {sum(COST.values()) / 60:.0f} projected minutes on this card "
      f"with one runner (a read about {cost('', 'steer') / 60:.1f} min)")

The stored engines come back from the Hub with their files (`modular_diffusion/capture/three_band/<job>/`; the first capture's clean engine from `modular_diffusion/capture/first_capture/`); the cell stops if one did not. One runner works the list in order. Each finished read is shipped at once to `REPO` under `modular_diffusion/capture/three_band/<job>/` (its `result.json` holds the read and the record of its part's weights), with the token from the notebook's secret `HF_TOKEN`. At the end the summary, the roster and its log and the instrument numbers are shipped to `_report_steering`. The machine is given back (`RELEASE`) only when every job and the report were shipped and no job failed; otherwise the cell says why and the machine is kept (run C3 again to resume).

In [ ]:
# C3 - the job list, each finished job shipped, a progress line a minute, the summary, the machine
from geolip.modular_diffusion import jobs
RELEASE = True                     # give the machine back at the end (only when every job and the report were shipped and no job failed)
EXTRA = ""                         # more arguments for every job (empty in a real run)
PROGRESS_S = 60

# this measurement's job list (the second cell's): the seven stored engines it reads come back from the Hub under the
# assert below and never run here (each from PREFIX, or from where HUB_AT names); what it runs is new
KEEP = [t for t, *_ in stored_jobs()]
for i, (t, a, ns, k) in enumerate(JOBS):      # every folder a job reads is a job before it
    assert all(n in [j[0] for j in JOBS[:i]] for n in ns), t
COST = {t: (0.0 if t in KEEP else cost(a, k)) for t, a, _, k in JOBS}
NEW = [t for t, *_ in JOBS if t not in KEEP]
assert len(NEW) == 6 + 6 + 4 and len(KEEP) == 7 and len({t for t, *_ in JOBS}) == len(JOBS), len(NEW)
print(f"this measurement: {len(NEW)} new jobs (reads of one part each), about {sum(COST.values()) / 60:.0f} projected "
      f"minutes on this card; {len(KEEP)} finished jobs come back from the Hub")

kind = {t: k for t, _, _, k in JOBS}
needs = {t: n for t, _, n, _ in JOBS}
tags = [t for t, *_ in JOBS]
roster = f"{OUT}/roster.txt"
cap = lambda t: max(30, int(3 * COST[t] / 60))
with open(roster, "w", encoding="utf-8", newline="\n") as fh:
    fh.write("\n".join(f"{t} | {cap(t)} | geolip.modular_diffusion.run | capture {a} {COMMON}{EXTRA}" + (f" | needs={','.join(n)}" if n else "")
                       for t, a, n, _ in JOBS) + "\nEND\n")
mark = lambda t, m: os.path.exists(f"{RUNS}/{t}/{m}")
if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
ENGINE_FILES = ("engine.safetensors", "engine_avg.safetensors", "engine.json", "train.json")
PICTURE_FILES = ("pictures.npz",)

def fetch_back():
    # on a new machine: finished jobs come back with their files, so nothing runs twice; a job comes back only when what
    # it reads came back too
    try:
        got = snapshot_download(repo_id=REPO, allow_patterns=[f"{HUB_AT.get(t, PREFIX)}/{t}/*" for t in tags], local_dir=f"{OUT}/hub_back")
    except Exception as e:
        print("nothing fetched back from the Hub:", repr(e)[:200])
        return
    for t in tags:
        src, dst = f"{got}/{HUB_AT.get(t, PREFIX)}/{t}", f"{RUNS}/{t}"
        if mark(t, "DONE") or not os.path.exists(f"{src}/result.json"):
            continue
        if kind[t] == "train" and not all(os.path.exists(f"{src}/{f}") for f in ENGINE_FILES):
            continue                                             # its engine files are not on the Hub: it is trained again
        if kind[t] == "pictures" and not all(os.path.exists(f"{src}/{f}") for f in PICTURE_FILES):
            continue                                             # its features are not on the Hub: it is run again
        if not all(mark(n, "DONE") for n in needs[t]):
            continue
        shutil.copytree(src, dst, dirs_exist_ok=True)
        open(f"{dst}/DONE", "w").write("fetched back from the Hub\n")
        open(f"{dst}/FROM_HUB", "w").write("fetched back from the Hub\n")

if REPO:
    fetch_back()
assert all(mark(t, "DONE") for t in KEEP), ("a job this measurement reads did not come back from the Hub: "
                                            + ", ".join(t for t in KEEP if not mark(t, "DONE")))
for t in tags:      # a claim left by an interrupted session would hold its job forever
    if mark(t, "CLAIM") and not (mark(t, "DONE") or mark(t, "FAILED")):
        os.remove(f"{RUNS}/{t}/CLAIM")
def after_job(tag, folder, ok):
    if ok and REPO:
        if hub.ship(folder, REPO, f"{PREFIX}/{tag}", f"the steering error of a part: {tag}", ignore=NOT_SHIPPED):
            open(f"{folder}/SHIPPED", "w").write("shipped to the Hub\n")      # kept across sessions on this machine
        else:
            print(f"{tag}: NOT SHIPPED (its folder stays on this machine: {folder})", flush=True)

shipped_or_fetched = lambda t: mark(t, "SHIPPED") or mark(t, "FROM_HUB")
for t in tags:      # a job that finished in an earlier session but whose upload failed: shipped again now
    if REPO and mark(t, "DONE") and not shipped_or_fetched(t):
        after_job(t, f"{RUNS}/{t}", True)

t0 = time.time()
left0 = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
print(f"start: {sum(mark(t, 'DONE') for t in tags)}/{len(tags)} jobs already done; about {left0 / 60:.0f} min to run; one runner")
runner = threading.Thread(target=jobs.run_roster, daemon=True,
                          kwargs=dict(roster=roster, out=RUNS, stop=f"{OUT}/STOP", log=f"{OUT}/roster.log", card_room_mib=None,
                                      ram_free_gb=4.0, max_hours=20.0, claim=True, after_job=after_job, wait_s=5.0,
                                      env={"OMP_NUM_THREADS": "4", "MKL_NUM_THREADS": "4"}))
runner.start()
while runner.is_alive():
    runner.join(PROGRESS_S)
    done, failed = sum(mark(t, "DONE") for t in tags), sum(mark(t, "FAILED") for t in tags)
    left = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
    spent = (time.time() - t0) / 60
    rate = (left0 - left) / 60 / spent if left0 > left and spent > 0 else 1.0     # projected minutes done a minute
    print(f"jobs done {done}/{len(tags)} (failed {failed}); {spent:.0f} min spent; about {left / 60 / rate:.0f} min left", flush=True)

summary = capture.summarize(RUNS)
print(summary)
REPORT = f"{OUT}/report"
os.makedirs(REPORT, exist_ok=True)
open(f"{REPORT}/summary.md", "w", encoding="utf-8").write(summary)
json.dump(INSTRUMENTS, open(f"{REPORT}/instruments.json", "w", encoding="utf-8"), indent=1)
for f in ("roster.txt", "roster.log"):
    if os.path.exists(f"{OUT}/{f}"):
        shutil.copy(f"{OUT}/{f}", f"{REPORT}/{os.path.basename(f).replace('.log', '_log.txt')}")
for t in tags:      # the logs of the jobs that failed
    if mark(t, "FAILED"):
        for f in ("run.log", "log.txt"):
            if os.path.exists(f"{RUNS}/{t}/{f}"):
                shutil.copy(f"{RUNS}/{t}/{f}", f"{REPORT}/failed_{t}_{f.replace('.', '_')}.txt")
shipped = hub.ship(REPORT, REPO, f"{PREFIX}/_report_steering", "the steering error of a part: summary, roster and checks") if REPO else None
unshipped = [t for t in tags if mark(t, "DONE") and not shipped_or_fetched(t)]
failed = [t for t in tags if mark(t, "FAILED")]
unfinished = [t for t in tags if not (mark(t, "DONE") or mark(t, "FAILED"))]
print("report shipped:", shipped, "| finished jobs not on the Hub:", ", ".join(unshipped) or "none", "| failed jobs:",
      ", ".join(failed) or "none", "| jobs not finished:", ", ".join(unfinished) or "none")
if RELEASE and shipped and not unshipped and not failed and not unfinished:
    from google.colab import runtime
    runtime.unassign()
else:
    why = (["the report was not shipped"] if not shipped else []) + (
        [f"these finished jobs were not shipped: {', '.join(unshipped)} (run C3 again to retry)"] if unshipped else []) + (
        [f"these jobs FAILED: {', '.join(failed)} (their logs are in the report)"] if failed else []) + (
        [f"these jobs did not finish: {', '.join(unfinished)} (run C3 again)"] if unfinished else [])
    print("THE MACHINE IS KEPT:", "; ".join(why) if why else "RELEASE is False")